# 12 — Contamination Probe: Counterfactual Entity Perturbation

WNUT-17 (2017) and SciERC (2018) are long-standing public benchmarks that were plausibly seen
during `gpt-4o-mini`'s pretraining, so the prompting arm's strong scores may partly reflect
**memorization of specific entity surface forms** rather than genuine context-based recognition.
This notebook tests that directly.

**Method.** We build a *counterfactual* copy of each target test set in which every gold entity
token is character-perturbed into a novel, unseen surface form (e.g. `Microsoft` → `Micronoft`),
while leaving all non-entity context, entity spans, and entity types **unchanged**. A model that
truly recognizes entities from context should degrade only modestly; a model relying on
memorized strings should collapse.

To separate memorization from the intrinsic difficulty of odd-looking tokens, we compare the
**LLM prompting arm** against a **fine-tuning control** (the budget-200 model) on the *same*
perturbed data. If the LLM's drop is much larger than fine-tuning's, that is evidence of
surface-form reliance; comparable drops indicate the perturbation is simply harder for everyone.

Original scores are read from the already-computed Arm 3 / Arm 1 results (no re-run needed);
only the perturbed evaluation is executed here.

In [1]:
!pip -q install "transformers==4.44.2" "datasets==2.19.2" "seqeval==1.2.2" "accelerate>=0.26.0" openai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


## Step 1 — Mount Drive, configure, keys

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, re, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED      = Path('/content/drive/MyDrive/AAI590/data/processed')
LABELS_DIR     = PROCESSED / 'label_maps'
MODELS_DIR     = PROCESSED / 'models'
RESULTS_DIR    = PROCESSED / 'results'
FEWSHOT_SPLITS = PROCESSED / 'fewshot_splits'
BASELINE_DIR   = MODELS_DIR / 'baseline_conll2003'

TARGET_DATASETS = ['wnut17', 'scierc']
BUDGET   = 200          # compare at the largest (strongest) budget
DEMO_SEED = 42
PERTURB_SEED = 42
MUTATION_RATE = 0.5     # fraction of alphabetic chars (after the first) mutated per entity token
EVAL_LIMIT = None       # None = full test set; set an int for a quick smoke test
INCLUDE_FT_CONTROL = True

out_dir = RESULTS_DIR / 'contamination'
out_dir.mkdir(parents=True, exist_ok=True)
PRED_DIR = out_dir / 'predictions'; PRED_DIR.mkdir(exist_ok=True)

# --- OpenAI key: paste temporarily for a quick run, then BLANK IT before committing ---
OPENAI_API_KEY = ""

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if not os.environ.get('OPENAI_API_KEY'):
    try:
        from google.colab import userdata
        if userdata.get('OPENAI_API_KEY'):
            os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception:
        pass
if not os.environ.get('OPENAI_API_KEY'):
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OPENAI_API_KEY (hidden): ')

OPENAI_MODEL = 'gpt-4o-mini'
MAX_OUTPUT_TOKENS = 512

def load_jsonl(p):
    rows = []
    with open(p) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

print('ready | model:', OPENAI_MODEL, '| budget:', BUDGET, '| FT control:', INCLUDE_FT_CONTROL)

Mounted at /content/drive
ready | model: gpt-4o-mini | budget: 200 | FT control: True


## Step 2 — Metrics (identical seqeval logic to Notebooks 05/09)

In [3]:
from seqeval.metrics import classification_report

def compute_entity_metrics(true_tags, pred_tags):
    rep = classification_report(true_tags, pred_tags, output_dict=True, zero_division=0)
    micro = rep['micro avg']
    per_type = {k: {'precision': float(v['precision']), 'recall': float(v['recall']),
                    'f1': float(v['f1-score']), 'support': int(v['support'])}
                for k, v in rep.items() if k not in ('micro avg', 'macro avg', 'weighted avg')}
    return {'typed_f1': float(micro['f1-score']), 'support': int(micro['support']), 'per_type': per_type}

## Step 3 — Counterfactual perturbation

Each entity token is character-mutated into a novel string of the **same length and casing**
(the first character is kept to preserve a capitalization cue), so the entity spans and BIO tags
are unchanged and only the surface form becomes unfamiliar. Non-entity tokens are untouched. The
perturbed test sets are saved for reproducibility.

In [4]:
def mutate_token(tok, rng):
    lower = 'abcdefghijklmnopqrstuvwxyz'; upper = lower.upper()
    out = []
    for i, ch in enumerate(tok):
        if ch.isalpha() and i > 0 and rng.random() < MUTATION_RATE:
            pool = lower if ch.islower() else upper
            repl = rng.choice(pool)
            out.append(repl if repl != ch else rng.choice(pool))
        else:
            out.append(ch)
    return ''.join(out)

def perturb_rows(rows, seed):
    rng = random.Random(seed)
    out = []
    for r in rows:
        new_tokens = [mutate_token(t, rng) if tag != 'O' else t
                      for t, tag in zip(r['tokens'], r['tags'])]
        out.append({**r, 'tokens': new_tokens, 'tags': r['tags']})
    return out

# build + save perturbed test sets
perturbed = {}
for ds in TARGET_DATASETS:
    orig = load_jsonl(PROCESSED / ds / f'{ds}_test.jsonl')
    pert = perturb_rows(orig, PERTURB_SEED)
    perturbed[ds] = pert
    with open(out_dir / f'{ds}_test_counterfactual.jsonl', 'w') as f:
        for r in pert:
            f.write(json.dumps(r, ensure_ascii=True) + '\n')
# quick before/after example
ex_o = load_jsonl(PROCESSED / 'wnut17' / 'wnut17_test.jsonl')
ex_p = perturbed['wnut17']
for o, p in zip(ex_o, ex_p):
    if any(t != 'O' for t in o['tags']):
        ent_o = [tok for tok, tg in zip(o['tokens'], o['tags']) if tg != 'O']
        ent_p = [tok for tok, tg in zip(p['tokens'], p['tags']) if tg != 'O']
        print('original entity tokens :', ent_o)
        print('perturbed entity tokens:', ent_p)
        break

original entity tokens : ['Sonmarg']
perturbed entity tokens: ['Soiearg']


## Step 4 — LLM prompting helpers (same prompt/parse as Notebook 09)

In [5]:
from seqeval.metrics.sequence_labeling import get_entities

TYPE_NOTES = {
    'wnut17': ('person, location, corporation, product, creative-work (songs, movies, books...), '
               'group (bands, sports teams...)'),
    'scierc': ('Task, Method, Metric, Material, Generic (a generic term like "approach" referring '
               'to a specific one), OtherScientificTerm'),
}
def entity_types_of(ds):
    tr = load_jsonl(PROCESSED / ds / f'{ds}_train.jsonl')
    return sorted({t.split('-', 1)[1] for r in tr for t in r['tags'] if t != 'O'})
def gold_entities(tokens, tags):
    return [{'text': ' '.join(tokens[s:e+1]), 'type': et} for et, s, e in get_entities(tags)]
def format_example(tokens, tags=None):
    line = 'Sentence: ' + ' '.join(tokens)
    return line + '\nEntities:' if tags is None else line + '\nEntities: ' + json.dumps(gold_entities(tokens, tags))
def build_system_prompt(ds, demo_rows):
    types = entity_types_of(ds)
    parts = ['You are a named entity recognition tagger.',
             f"Entity types for this domain: {', '.join(types)}.",
             f"Type hints: {TYPE_NOTES[ds]}.",
             'For the given sentence, list every entity as a JSON array of objects with keys '
             '"text" (the exact contiguous span, copied verbatim) and "type" (one of the types above). '
             'Preserve order of appearance. If there are no entities, answer []. Answer with ONLY the JSON array.']
    if demo_rows:
        parts += ['', f'Here are {len(demo_rows)} labeled examples from this domain:', '']
        parts += [format_example(r['tokens'], r['tags']) for r in demo_rows]
    return '\n'.join(parts)
def parse_entities(text):
    try:
        out = json.loads(text); return out if isinstance(out, list) else None
    except json.JSONDecodeError:
        pass
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if m:
        try:
            out = json.loads(m.group(0)); return out if isinstance(out, list) else None
        except json.JSONDecodeError:
            return None
    return None
def entities_to_bio(tokens, entities, allowed):
    tags = ['O'] * len(tokens); lower = [t.lower() for t in tokens]
    canon = {t.lower(): t for t in allowed}
    for ent in entities or []:
        if not isinstance(ent, dict) or 'text' not in ent or 'type' not in ent: continue
        et = canon.get(str(ent['type']).lower())
        if et is None: continue
        span = str(ent['text']).split()
        if not span: continue
        placed = False
        for exact in (True, False):
            hay = tokens if exact else lower
            needle = span if exact else [w.lower() for w in span]
            for i in range(len(tokens) - len(span) + 1):
                if hay[i:i+len(span)] == needle and all(t == 'O' for t in tags[i:i+len(span)]):
                    tags[i] = f'B-{et}'
                    for k in range(i+1, i+len(span)): tags[k] = f'I-{et}'
                    placed = True; break
            if placed: break
    return tags

In [6]:
import openai
from openai import OpenAI
client = OpenAI()

def _create_with_retry(**kw):
    delay = 1.0
    for _ in range(8):
        try:
            return client.chat.completions.create(**kw)
        except openai.RateLimitError as e:
            if 'insufficient_quota' in str(e): raise
            time.sleep(delay); delay = min(30.0, delay*2)
        except (openai.APITimeoutError, openai.APIConnectionError, openai.InternalServerError):
            time.sleep(delay); delay = min(30.0, delay*2)
    return client.chat.completions.create(**kw)

def llm_predict(ds, eval_rows, tag):
    demos = load_jsonl(FEWSHOT_SPLITS / ds / f'{ds}_train_{BUDGET}_seed_{DEMO_SEED}.jsonl')
    system_prompt = build_system_prompt(ds, demos)   # demos are the ORIGINAL (unperturbed) examples
    allowed = entity_types_of(ds)
    pred_fp = PRED_DIR / f'{ds}_{tag}.jsonl'
    cached = {r['id']: r for r in load_jsonl(pred_fp)} if pred_fp.exists() else {}
    gold, pred = [], []
    for i, row in enumerate(eval_rows):
        if row['id'] in cached:
            rec = cached[row['id']]
        else:
            raw = ''.join(b for b in [ _create_with_retry(model=OPENAI_MODEL, max_tokens=MAX_OUTPUT_TOKENS,
                    temperature=0, messages=[{'role':'system','content':system_prompt},
                    {'role':'user','content':format_example(row['tokens'])}]).choices[0].message.content or '' ])
            tags = entities_to_bio(row['tokens'], parse_entities(raw), allowed)
            rec = {'id': row['id'], 'pred_tags': tags}
            with open(pred_fp, 'a') as f: f.write(json.dumps(rec, ensure_ascii=True) + '\n')
        gold.append(row['tags']); pred.append(rec['pred_tags'])
        if (i+1) % 100 == 0: print(f'    {ds} {tag}: {i+1}/{len(eval_rows)}')
    return compute_entity_metrics(gold, pred)

## Step 5 — Run the LLM on the counterfactual test sets

In [7]:
# original LLM scores come from Notebook 09's results (no re-run needed)
llm_orig_csv = RESULTS_DIR / 'llm_prompting' / 'llm_prompting_results.csv'
llm_orig = pd.read_csv(llm_orig_csv)

rows = []
for ds in TARGET_DATASETS:
    o = llm_orig[(llm_orig.dataset == ds) & (llm_orig.budget == BUDGET) & (llm_orig.provider == 'openai')].iloc[0]
    orig_f1 = float(o['test_typed_f1']); orig_per_type = json.loads(o['per_type'])
    pert_rows = perturbed[ds][:EVAL_LIMIT] if EVAL_LIMIT else perturbed[ds]
    print(f'== LLM on perturbed {ds} ({len(pert_rows)} sentences) ==')
    m = llm_predict(ds, pert_rows, tag='cf')
    rows.append({'dataset': ds, 'method': 'LLM prompting', 'orig_f1': round(orig_f1, 3),
                 'perturbed_f1': round(m['typed_f1'], 3), 'drop': round(orig_f1 - m['typed_f1'], 3)})
    # stash per-type for later
    m['orig_per_type'] = orig_per_type
    perturbed[ds + '_llm'] = m
llm_df = pd.DataFrame(rows)
display(llm_df)

== LLM on perturbed wnut17 (1287 sentences) ==
    wnut17 cf: 100/1287
    wnut17 cf: 200/1287
    wnut17 cf: 300/1287
    wnut17 cf: 400/1287
    wnut17 cf: 500/1287
    wnut17 cf: 600/1287
    wnut17 cf: 700/1287
    wnut17 cf: 800/1287
    wnut17 cf: 900/1287
    wnut17 cf: 1000/1287
    wnut17 cf: 1100/1287
    wnut17 cf: 1200/1287
== LLM on perturbed scierc (551 sentences) ==
    scierc cf: 100/551
    scierc cf: 200/551
    scierc cf: 300/551
    scierc cf: 400/551
    scierc cf: 500/551


,dataset,method,orig_f1,perturbed_f1,drop
0,wnut17,LLM prompting,0.462,0.186,0.276
1,scierc,LLM prompting,0.435,0.024,0.411


## Step 6 — Fine-tuning control (budget-200 model on original vs perturbed)

We retrain a single few-shot model (budget 200, seed 42) with the same recipe as Notebook 05 and
evaluate it on both the original and perturbed test sets, so its drop is measured on the very same
data as the LLM's. Set `INCLUDE_FT_CONTROL = False` in Step 1 to skip.

In [8]:
if INCLUDE_FT_CONTROL:
    import torch
    from transformers import (AutoConfig, AutoModelForTokenClassification, AutoModelForMaskedLM,
                              AutoTokenizer, DataCollatorForTokenClassification, Trainer,
                              TrainingArguments, set_seed)
    from datasets import Dataset

    def load_label_map(ds):
        m = json.load(open(LABELS_DIR / f'{ds}_label_map.json'))
        return ({str(k): int(v) for k, v in m['label2id'].items()},
                {int(k): str(v) for k, v in m['id2label'].items()})

    def tokenize_align(rows, tokenizer, label2id, max_length=256):
        def enc(batch):
            e = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True, max_length=max_length)
            labels = []
            for i, tags in enumerate(batch['tags']):
                wids = e.word_ids(batch_index=i); prev = None; lab = []
                for w in wids:
                    lab.append(-100 if (w is None or w == prev) else label2id[tags[w]]); prev = w
                labels.append(lab)
            e['labels'] = labels; return e
        return Dataset.from_list([{'tokens': r['tokens'], 'tags': r['tags']} for r in rows]) \
            .map(enc, batched=True, remove_columns=['tokens', 'tags'])

    def predict_tags(model, tokenizer, rows, id2label, max_length=256, bs=32):
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'; model = model.to(dev).eval(); out = []
        for s in range(0, len(rows), bs):
            batch = rows[s:s+bs]
            e = tokenizer([r['tokens'] for r in batch], is_split_into_words=True, truncation=True,
                          max_length=max_length, padding=True, return_tensors='pt')
            with torch.no_grad():
                logits = model(**{k: v.to(dev) for k, v in e.items()}).logits.cpu()
            preds = logits.argmax(-1)
            for i, r in enumerate(batch):
                wids = e.word_ids(batch_index=i); prev = None; tags = []
                for pos, w in enumerate(wids):
                    if w is not None and w != prev: tags.append(id2label[int(preds[i][pos])])
                    prev = w
                while len(tags) < len(r['tokens']): tags.append('O')
                out.append(tags)
        return out

    baseline = AutoModelForTokenClassification.from_pretrained(str(BASELINE_DIR))
    tokenizer = AutoTokenizer.from_pretrained(str(BASELINE_DIR))
    collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    ft_rows = []
    for ds in TARGET_DATASETS:
        label2id, id2label = load_label_map(ds)
        cfg = AutoConfig.from_pretrained('bert-base-cased', num_labels=len(id2label),
                                         id2label=id2label, label2id=label2id)
        model = AutoModelForTokenClassification.from_pretrained('bert-base-cased', config=cfg,
                                                                ignore_mismatched_sizes=True)
        model.base_model.load_state_dict(baseline.base_model.state_dict(), strict=True)
        set_seed(DEMO_SEED)
        train_rows = load_jsonl(FEWSHOT_SPLITS / ds / f'{ds}_train_{BUDGET}_seed_{DEMO_SEED}.jsonl')
        train_ds = tokenize_align(train_rows, tokenizer, label2id)
        args = TrainingArguments(output_dir=f'/content/ft_ctrl_{ds}', num_train_epochs=8,
                                 per_device_train_batch_size=8, learning_rate=2e-5, weight_decay=0.01,
                                 logging_strategy='no', save_strategy='no', report_to='none', seed=DEMO_SEED)
        Trainer(model=model, args=args, train_dataset=train_ds, tokenizer=tokenizer,
                data_collator=collator).train()

        orig = load_jsonl(PROCESSED / ds / f'{ds}_test.jsonl')
        pert = perturbed[ds]
        if EVAL_LIMIT: orig, pert = orig[:EVAL_LIMIT], pert[:EVAL_LIMIT]
        mo = compute_entity_metrics([r['tags'] for r in orig], predict_tags(model, tokenizer, orig, id2label))
        mp = compute_entity_metrics([r['tags'] for r in pert], predict_tags(model, tokenizer, pert, id2label))
        ft_rows.append({'dataset': ds, 'method': 'Fine-tuning', 'orig_f1': round(mo['typed_f1'], 3),
                        'perturbed_f1': round(mp['typed_f1'], 3), 'drop': round(mo['typed_f1'] - mp['typed_f1'], 3)})
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    ft_df = pd.DataFrame(ft_rows)
    display(ft_df)
else:
    ft_df = None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss


,dataset,method,orig_f1,perturbed_f1,drop
0,wnut17,Fine-tuning,0.376,0.255,0.121
1,scierc,Fine-tuning,0.371,0.208,0.163


## Step 7 — Verdict: whose accuracy survives entity perturbation?

In [9]:
final = pd.concat([llm_df] + ([ft_df] if ft_df is not None else []), ignore_index=True)
final = final[['dataset', 'method', 'orig_f1', 'perturbed_f1', 'drop']].sort_values(['dataset', 'method'])
final.to_csv(out_dir / 'contamination_summary.csv', index=False)
display(final)
print('\nInterpretation: a LARGER drop for the LLM than for fine-tuning on the same perturbed data')
print('indicates reliance on memorized/known entity surface forms (a contamination signal);')
print('comparable drops indicate the perturbation is simply harder for both, not LLM-specific memorization.')
print('\nSaved -> ', out_dir / 'contamination_summary.csv')

,dataset,method,orig_f1,perturbed_f1,drop
3,scierc,Fine-tuning,0.371,0.208,0.163
1,scierc,LLM prompting,0.435,0.024,0.411
2,wnut17,Fine-tuning,0.376,0.255,0.121
0,wnut17,LLM prompting,0.462,0.186,0.276



Interpretation: a LARGER drop for the LLM than for fine-tuning on the same perturbed data
indicates reliance on memorized/known entity surface forms (a contamination signal);
comparable drops indicate the perturbation is simply harder for both, not LLM-specific memorization.

Saved ->  /content/drive/MyDrive/AAI590/data/processed/results/contamination/contamination_summary.csv
